# Dataset integration

In this tutorial, we will integrate a differential centrifugation (DC) dataset with an organellar immunoprecipitation (OrgIP) dataset.

While DC is a more scalable technique, OrgIP provides significantly higher resolution. A practical approach to achieving high-resolution mapping is to strategically supplement DC data with targeted IP pulldowns for cellular compartments that are poorly resolved by centrifugation alone. 

Although we merge two complete datasets in this example, real-world applications typically involve combining a DC dataset with only a small number of strategic IPs. 

This integration process is streamlined using the grassp package.

If you have not installed *grassp* yet, please follow the [installation instructions](https://czbiohub-sf.github.io/grassp/installation.html).

In [ ]:
import grassp as gr
import scanpy as sc
import anndata as ad
import numpy as np

## Loading Data


In [ ]:
# load DC dataset (hek cells)
dc = gr.ds.load_dataset("Frank_2026_HEK293")
dc

In [ ]:
# OrgIP dataset (hek cells)
orgip = gr.ds.load_dataset("Hein_2025_HEK293_WT")
# This object ships the untagged (WT) control pulldowns and unscaled log fold
# changes. Drop the controls and z-score each bait, so that the ~70 IP columns
# and the 7 DC fractions contribute on a comparable scale when we concatenate.
orgip = orgip[:, ~orgip.var_names.str.endswith("-WT")].copy()
sc.pp.scale(orgip)
orgip

## Plotting individual maps

Next, we use UMAP to visualize the spatial organization of the datasets. This helps us qualitatively assess how well the biological compartments are resolved before proceeding with downstream integration or analysis.

Note that we can plot right away, without calling `sc.pp.neighbors` and `sc.tl.umap` first: datasets from the grassp portal ship with a precomputed embedding in `.obsm["X_umap"]`, and that is what `sc.pl.umap` draws. Those coordinates were computed for the full published map, so dropping the untagged controls and z-scoring the OrgIP object above does not change them. For the integrated object further down — and for your own data — we have to compute the embedding ourselves first.

In [ ]:
sc.pl.umap(dc, color="marker_hein2025_gt", title="DC map")
sc.pl.umap(orgip, color="marker_hein2025_gt", title="OrgIP map")

## Integration

To integrate the two datasets, we must identify a common gene or protein identifier to facilitate matching. While UniProt IDs or gene names are typical candidates, direct merging via UniProt IDs is challenging in this case because one dataset was searched against Swiss-Prot and the other against TrEMBL. To ensure consistency, we will first collapse all entries based on their gene names.

In [ ]:
# aggregate proteins in the DC dataset
dc = dc[dc.obs["Gene names"].notna()]
dc_agg = gr.pp.aggregate_proteins(dc, grouping_columns="Gene names", agg_func=np.mean)
dc_agg.obs.head()

In [ ]:
# aggregate proteins in the OrgIP dataset
orgip_agg = gr.pp.aggregate_proteins(
    orgip, grouping_columns="Gene_name_canonical", agg_func=np.mean
)
orgip_agg.obs_names = orgip_agg.obs_names.str.upper()
orgip_agg.obs.head()

In [ ]:
# combine the two datasets, keep only the common proteins,
combined = ad.concat(
    [dc_agg, orgip_agg],
    axis=1,  # This specifies a horizontal concatenation. We are keeping the proteins as rows and concatenating the experimental fractions or pulldowns from both datasets as columns.
    join="inner",
    merge="first",
    keys=["dc", "orgip"],
    label="dataset",
)
combined

In [ ]:
# compute UMAP embeddings
sc.pp.neighbors(combined, use_rep="X")
sc.tl.umap(combined, min_dist=0.1)

In [ ]:
# plot UMAP of the integrated dataset
sc.pl.umap(combined, color="marker_hein2025_gt")